<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 1: Character Level Language Model (CLLM)

#### Tim Moroney, 2026


A lesson where we build a character level language model.  It's kind of like chat GPT except it can't chat and doesn't use transformers.  It does generate English text though: one _character_ at a time!


# Reference text
Throughout these notes we will sometimes make reference to the excellent book _Algorithms for Optimization_ by Mykel J. Kochenderfer and Tim A. Wheeler.
The second edition of this book is highly recommended, and available for free download from the website
https://algorithmsbook.com/optimization/

# Jupyter notebooks
This document is a so-called _Jupyter notebook_.  The name derives from three programming languages: _Ju_ stands for _Julia_, _pyt_ for _Python_, and _"er"_ for _R_.  Perhaps you have used some or all of these languages before. If so, you may even be familiar with this Jupyter environment.  It is certainly convenient that all three languages, which are so important to mathematics, statistics and machine learning, can be unified in a common notebook format.

The code in this particular notebook is written in the _Julia_ language. Of the three choices, _Julia_ is the most ergonomic for scientific computing, and indeed it was explicitly designed as a language for such.  In the first few notebooks in this series, we will find that we can build up performant, mini-AI models in Julia, from first principles, without sacrificing clarity.

Later in the series, we will demonstrate how to use _deep learning libraries_ to build more complex models.  In that case, both Julia and Python are equally convenient, and we will present examples using both.

Note: in Google Colab, the language and environment is automatically detected from the notebook.  If you are starting a new notebook you can switch between languages by choosing Runtime, Change Runtime type, and choose between the three languages from the drop-down list.  

# The Julia language
If you have not used Julia previously, before proceeding further in this notebook you should download the book [Algorithms for Optimization](https://algorithmsbook.com/optimization/) (Second Edition) and read _Appendix A: Julia_.  In particular, the following sections of that Appendix are especially relevant:

* A.1 Types: A.1.1 -- A.1.7

* A.2 Functions: A.2.1 -- A.2.4

* A.3 Control Flow: A.3.1 -- A.3.2

# Package management

(There is no need to study this code.)
We start by installing the required packages.  If you're running on Colab this will download pre-compiled code, expect about a minute.  Otherwise be prepared to wait a little longer for installation the first time you run. Feel free to read ahead while you wait.

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# Large Language Models

You will have heard that generative AI tools such as Chat GPT are powered by so-called large language models (LLMs).  Beneath all the hype, an LLM is just a mathematical function that takes a string of text as input, and outputs a prediction for what should come next.  It's literally a "next token predictor".

Modern LLMs are truly large, in the sense that the mathematical function they use is a very deep neural network with many billions of parameters. Suitable values for these parameters are learned through a training process which iteratively adjusts the parameter values to improve the model's performance at making predictions for an enormous dataset of training text.

More technically, an LLM is an **autoregressive** process: each new token is predicted conditionally on the values of all the previous tokens.  Once a new token is predicted, it becomes one of the previous tokens that condition the prediction of the next.  And so on, one token at a time.  It's remarkable that next token prediction, at scale, is the entire basis for the "intelligence" that you may (or may not) ascribe to a modern AI chatbot.  

In the first few lessons of this unit, we will learn all of the important mathematics that makes LLMs possible: neural networks, back propagation, gradient descent, and more; as well as other ideas that are important for AI models at smaller scales.

Our philosophy is to present the mathematical techniques as they are required.  So we will commence building our own model, and see what maths falls out along the way that we need to understand.

But it will be a toy model.  An actual LLM is beyond our scope to build from scratch: it would require far too much computing resources to adequately train (although we will inspect a pre-trained model in a later lesson).

Instead, we will focus on building a character-level language model (CLLM): a toy version of an LLM that predicts next _characters_ only.  It won't be able to chat to you.  But, properly trained, it can produce plausible "English-looking" text; albeit with no semantic content whatsoever.




# On to the building the model

Our character level language model (CLLM) takes the form of a parameterised model $M_p(u)$.  Its job is to take as input an array of characters, represented (in a way to be clarified) by the vector $u$, and produce an output (in a way to be clarified) that predicts the next character in the sequence.  

The way it learns to make predictions is by **training** on a suitable set of English language text.  By observing many real sequences of text and observing what characters tend to follow others, it **learns** suitable values of the parameters $p$ (in a way to be clarified) so that its predictions are consistent with real English text.

For example, if the input vector $u$ represents (somehow) the text `"alice said"`, then a good prediction for the next character would be `' '`, the space character.  A bad prediction would be `'q'` because the text `"alice saidq"` is not common in standard English.

Lots of questions present themselves now.
* What possible characters are allowed in the input text?
* How are they represented by the input vector $u$?
* What is the functional form of $M_p(u)$?
* How does the model communicate its prediction for the next character?
* What is the role and nature of the parameters $p$?

We will work through answers to these questions throughout the rest of this lesson.

# Define the model's vocabulary
We begin by defining the vocabulary $\mathcal{V}$ our CLLM will understand.

Since our model deals only in characters, its vocabulary will be just a set of characters. For simplicity we'll just use lower case English characters and space.
$$
\mathcal{V} = \{\texttt{'a'},\ \texttt{'b'},\ \ldots,\ \texttt{'z'},\ \texttt{' '}\}
$$
with vocabulary size
$$
|\mathcal{V}| = 27.
$$

In code, our model's vocabulary $\mathcal{V}$ will be the 27 elements of the array `chars`.

In [ ]:
# Define the character set
chars = ['a':'z'; ' ']      # we only deal with lowercase text and space
print(chars)

vocab_size = length(chars)  # number of characters in our "vocabulary"

# Encoder/decoder functions

We will choose to represent characters by their index in this vocabulary.  So `'a'` $\Leftrightarrow 1,$ `'b'` $\Leftrightarrow 2,$ etc.  We can write little functions that convert between an actual character and its index, which you can think of encoding and decoding between characters and their numeric representations.

### Mapping index to character

This is just looking up the `i`th entry of `chars`.  For example, index 16 maps to character`'p'`.

In [ ]:
# Define the function
idx_to_char(i) = chars[i]

# Give it a test
idx_to_char(16)

### Mapping character to index

This just finding the index of the character in the array `chars`.  For example, character `'p'` maps to index 16.

In [ ]:
# Define the function
char_to_idx(c) = findfirst(isequal(c), chars)

# Give it a test
char_to_idx('p')

### Mapping string to indices

Mapping an entire string to an array of indices is just applying `char_to_idx` to every character in the string.  For example, the text `"alice said"` can be represented as the array of indices shown below.



In [ ]:
# Define the function
string_to_idxs(s) = [char_to_idx(c) for c in s]

# Give it a test
u = string_to_idxs("alice said")

# Model input and output

The input vector $u$ to the model function $M_p(u)$ will be a vector of integers, representing a sequence of text.  For example, $u$ could be the vector of integers we created just before
$$
u = \left[\begin{array}{c}1\\12\\9\\3\\5\\27\\19\\1\\9\\4\end{array}\right] \in \{1,\ldots,|\mathcal{V}| \}^{C}
$$
representing the text `"alice said"`.  Note that in our model, the size of the input vector is fixed -- this is known as the model's _context size_, which we denote as $C$.  Real LLMs have context sizes that are many thousands of tokens.  Our simple model will have a context size of just $C = 10$ characters.

The model must then try to predict what the next character in the text sequence should be. The best prediction for the next character after the sequence `"alice said"` would presumably be the space character. So should we expect the output of the model is $M_p(u) = 27$?

Actually, no. That's not quite how autoregressive models work. They don't output a _definitive_ prediction for the next character (or next token, in real LLMs). Instead they output _probabilities_ for every possible next character (or next token) conditional on the input.


So in our model, that looks like a 27-dimensional vector ($|\mathcal{V}| = 27$) with nonnegative entries that sum to 1.  For the particular model we'll run today, the actual output for this particular input $u$ is
$$
\hat{y} = M_p(u) = \left[\begin{array}{c}0.001\\0.004\\0.003\\0.011\\\vdots\\0.000\\0.000\\0.001\\0.822\end{array}\right] \in \mathbb{R}^{|\mathcal{V}|}
$$

And indeed the probability corresponding to the space character, which is the final entry in the output vector, is by far the largest, at $0.822$.  The rest of the probability is shared over the other 26 characters; the next most likely character happens to be `'s'`, with probability $0.061$ and then `'l'` with probability $0.018$.

To generate new text using the model, we would use these probabilities to _sample_ the next character in the sequence -- meaning there is inherent randomness in the prediction. On most runs we would sample space to come next, but on some runs with the same input we may instead sample `'s'`, for example.

So to summarise, our model's input is a vector $u \in \{1,\ldots,|\mathcal{V}| \}^{C}$ representing the sequence of text we want to continue, in the form of character indices.  And our model's output is $\hat{y} = M_p(u) \in \mathbb{R}^{|\mathcal{V}|}$ representing the conditional probabilities

$$
\hat{y}_i = \textrm{Prob}(\textrm{next char} = i\ |\ u;p)
$$

that the model has assigned to each possible next character in the sequence. The output always satisfies $\hat{y}_i \geq 0$ and $\sum_{i} \hat{y}_i = 1$ so it represents a valid probability distribution for the next character.

# Model parameters

Now what about the parameters $p$? The whole premise of machine learning is that you can design models that have many parameters, and then learn from lots of training examples what suitable values of these parameters should be, so that the model is effective at making predictions.

There are limitless possible parameterised models you might dream up, but most AI models are built from the same basic ingredients. And these ingredients are mostly matrix and vector arithmetic operations. By sticking with these basic building blocks, AI models are able to be trained and evaluated using specialised hardware (like GPUs), and are found to work well in practice.  (There is some mathematical justification behind the claim that neural networks in particular work well in practice, in the form of the [universal approximation theorem](https://https://en.wikipedia.org/wiki/Universal_approximation_theorem).  We don't plan to dwell on this point however.)

In our simple CLLM, the learnable parameters will comprise three matrices and two vectors.  The matrices, which we'll denote $W_e, W_1$ and $W_2$ are referred to as "weights".  And the vectors, which we'll denote $b_1$ and $b_2$ are referred to as "biases".

Notationally we bundle all these weights and biases into a single symbol $p$ and denote our parameterised model simply as $M_p$.  You can think of the symbol $p$ as representing a "vector of parameters"

$$p = [W_e, W_1, W_2, b_1, b_2],$$

even though each entry of this parameter vector is itself either a matrix or vector.  When we write $\hat{y} = M_p(u)$ we mean that the model is using the particular weights and biases given by the parameter vector $p$ to compute $\hat{y}$ as a function of $u$.

Another day we will discuss how to get the model to learn suitable values for the parameters by training on lots of sample data. But for now, let's just load a suitable parameter vector $p$ that was prepared in this way.  The training data used in this case was the full text of the book _Alice's Adventures in Wonderland_.

## Loading pre-trained parameters

We download a workspace with pre-trained parameters.  From the output of the code below you can see that `p` is precisely the "vector of parameters" we mentioned earlier. Each component of `p` is either a matrix of weights or a vector of biases.

In [ ]:
# Load some pre-trained parameters for this model
url = "https://github.com/moroneyt/MXB301/raw/main/resources/CLLM_pretrained.jld2"
paramfile = jldopen(download(url))
p = paramfile["p"]
close(paramfile)
p # let's see what we have here

## Component Vectors

The variable `p` is a `ComponentVector` so we can pick out its constituent components.  For example, here is the weight matrix $W_1$:

In [ ]:
p.W1

# Model implementation

We continue our discussion of the model by examining the complete code to evaluate $M_p(u)$.  There are six lines of code, which involve only indexing, matrix-vector arithmetic, and a couple of nonlinear functions `tanh` and `softmax`.

## Model code
Our AI model is an example of a [neural network](https://en.wikipedia.org/wiki/Neural_network_(machine_learning)).  It's common to describe neural networks as comprised of _layers_. In ths terminology, our model comprises an embedding layer, followed by two dense layers, and a softmax output layer. All of this we will explain in detail in a moment. But for now observe that it's all just basic matrix-vector operations and a couple of nonlinear functions.  If we wanted to, we could write it all out as a one-line formula, which looks a bit peculiar, but just for fun, that formula is

$$M_p(u) = \textrm{softmax}(W_2\textrm{tanh}.(W_1\textrm{vec}(W_e[:,u])+b_1 )+b_2).$$

That's genuinely all there is to our model.

Here it is in code.  We have called the function `forward` in our code for reasons of convention, which will become clearer in later weeks.


In [ ]:
# Predict probabilities of the next character
function forward(u, p)
    X = p.We[:, u]               # 1. embedding
    v = vec(X)                   # 2. flatten
    z1 = p.W1 * v  + p.b1        # 3. first dense layer
    h1 = tanh.(z1)               # 4. activation
    z2 = p.W2 * h1 + p.b2        # 5. second dense layer
    ŷ = softmax(z2)              # 6. softmax for probabilities
    return ŷ
end

## Fixing the parameters

The `forward` function takes both `u` and `p` as inputs.  These represent the text to continue, and the model parameters respectively.  When applying the model, it's sometimes convenient to just fix the parameters at their trained values, and consider the model as a function of `u` only.  We'll define a new function called `model` that does exactly this.

In [ ]:
model(u) = forward(u, p)  # fix the parameters p to their loaded values in our workspace

## Example

Before analysing each component of the model in detail, it's surely tempting to try it out on an input and see what it can do.

We can input our array `u` from earlier, corresponding to the text `"alice said"`.

In [ ]:
ŷ = model(u)

## Deterministic sampling using argmax

The output is a vector of the predicted probabilities for the next character in the sequence. As already noted previously, the space character is by far the most likely in this case.  If we want to be _deterministic_ when choosing the next character, we would just choose whichever has the largest probability: that's a job for `argmax`.  (Later on we will actually sample randomly, so the predicted next character wouldn't necessarily have to be space, although it would be the most likely choice by far.)

In [ ]:
idx = argmax(ŷ)  # the index of the maximum entry

## From index to character

Index 27 of course corresponds to the space character, which we can confirm using `idx_to_char`.

So for this example, the model seems to do a good job at inferring that the most likely next character in the sequence is space.

In [ ]:
idx_to_char(idx) # the character it represents

# Embedding layer

We now turn to the detailed description of the calculations that the model performs in each layer, so that we can completely understand how it works.

The first layer of our model is the _embedding layer_.  It is parameterised by the embedding matrix $W_e$. You recall that the input to the model, and hence to this first layer, is the vector $u \in \{1,\ldots,|\mathcal{V}| \}^{C}$ of character indices, representing the sequence of text that we want to continue. The role of the embedding layer is to map each character index to a point in "embedding space".

The idea of embedding is central to natural language processing. It turns out that inside the neural network, it's not satisfactory to just use the raw indices to represent characters.  The problem is that a simple mapping like `'a'` $\Leftrightarrow 1,$ `'p'` $\Leftrightarrow 16,$ `'z'` $\Leftrightarrow 26,$ `' '` $\Leftrightarrow 27$ doesn't capture any information about the different roles that those characters play in English text.  Clearly the space character is not actually "one more than `'z'`" in any meaningful sense, and a `'p'` isn't worth 16 times an `'a'` or anything.

Instead, we want our model to learn a richer way to embed usage patterns into the representation of each character.  A way that would treat a space character as different from the other characters, that might distinguish vowels from consonants, that might account for other idiosyncrasies of English, like `'x'` rarely beginning a word, `'j'` almost always beginning a word, and so on.  The embedding matrix $W_e$ is the way of encoding this richer level of information in the representation of each character.  Each column of $W_e \in \mathbb{R}^{d_e \times |\mathcal{V}|}$ represents the embedding of a character into $d_e$-dimensional _embedding space_.



## Embedding matrix

Having said that, let's take a look at the embedding matrix that the model learned when it was trained.

You can see that $W_e \in \mathbb{R}^{2 \times 27}$.  That is, the embedding space is two-dimensional: $d_e = 2$.  To be clear, the _dimension_ of the space was not learned; it was specified at the outset, and chosen to be two-dimensional for ease of visualisation.  It is the entries of the matrix $W_e$ that are learned.

In [ ]:
p.We  # the embedding matrix

## Coordinates in embedding space

For example, the coordinates of `'q'` in this two-dimensional embedding space are

In [ ]:
p.We[:,17]  # coordinates of 'q'

## Dimension of embedding space

The dimension $d_e$ of the embedding space is a design choice when building the model. In real LLMs the embedding space dimension is _much_ larger, perhaps in the thousands for a GPT type of model.  But that's an embedding space for _tokens_, for which there is much more semantic context, and high-dimensional space makes sense to try to capture as much of that meaning as possible. For our character-based model there is no need for such a high-dimensional embedding space.

Remember that the actual coordinates of each character in embedding space are not programmed by anyone.  Instead they are learned during the training process.  And the particular coordinates learned are just whatever they need to be for the model to do well at predicting new characters.

So with that in mind, let's now _visualise_ the learned character embedding for this model. Because we chose to use a two-dimensional embedding space, we can just plot each character's coordinates in the plane.

In [ ]:
# Visualise the learned character embeddings
fig, ax = scatter(p.We)
text!(p.We, text=string.(chars), align=(:left, :top), font=:italic, fontsize=16)
ax.title = "Learned character embeddings"
fig

Sure enough, the model has learned to associate the space character with a coordinate that is clearly distinct from all the others. Likewise the vowels are rather distinct from the consonants, with `'i'` in particular far removed from the rest, while `'a'` and `'u'` are very close to one another. There is some differentiation amongst the consonants, with the likes of `'j'`, `'q'`, `'x'`, `'z'`, and `'v'` somewhat separated from the main bunch.

Again we emphasise that whatever meaning is encoded in these coordinates is just a side-effect from the actual objective the model was trained for, which is predicting the next character in a sequence of text. It turns out that in solving that task, the most useful way to embed characters in two-dimensional space does indeed correspond to intuitive notions we have about the different roles that characters play in English text.

## Embedding code

We can now fully understand the first line of the model code.

`X = p.We[:, u]`

Given an input vector of character indices $u$, it simply picks out those corresponding columns of $W_e$; i.e. the coordinates of those characters in embedding space.

The resulting matrix $X \in \mathbb{R}^{d_e \times C} = \mathbb{R}^{2 \times 10}$ has, as its columns, the two-dimensional coordinates ($d_e = 2$) corresponding to the 10 input characters in the context ($C = 10$).

In [ ]:
X = p.We[:, u]

# Flattening layer
The second line of code in the model is a flattening operation

`
v = vec(X)
`

which simply flattens (or `vec`torises) the $d_e\times C$ matrix $X$ into one long vector $v \in \mathbb{R}^{d_eC}$.  The vector $v$ has the coordinates of the first input character (`'a'` in our example) in its first two entries, the coordinates of the second input character (`'l'` in our example) in its next two entries, and so on.

The next layer will take this vectorised representation as its input.  The fact that the values in this vector correspond to ordered pairs of character coordinates is not directly communicated.  Instead, the next layer will have learned whatever it needs to do with all this information itself during training.

In [ ]:
v = vec(X)

# First dense layer

Next comes the first dense layer.  This type of layer is ubiquitous in deep learning (also commonly referred to as a **fully connected layer**), and it consists of a matrix-vector product and a vector addition

$$
z_1 = W_1\,  v + b_1\,.
$$

Given that the input vector to this layer $v$ has a known size (20, in this example), the only design choice to make about the layer is the size of the output $z_1$ -- that is, the first dimension of $W_1$ and the dimension of $b_1$ (which must match).  As is clear from the output below, we chose (rather arbitrarily) to set this so-called _hidden dimension_ to be 16.  So $W_1 \in \mathbb{R}^{16 \times 20}$ and $b_1 \in \mathbb{R}^{16}$.

In [ ]:
z1 = p.W1 * v + p.b1  # matrix-vector product and vector addition

# The neural analogy

The original motivation for this type of layer, and indeed the origin of the name _neural network_ was that the input vector and output vector were interpreted as layers of "neurons" (as in the brain), and the weights in the matrix $W_1$ were the strengths of the connections between them (mirroring actual biological synapses).  So the "signal" received by one layer of neurons was a weighted combination of the "activations" of the neurons in the previous layer. The bias $b_1$ was then added, analogous to a background excitation level of each neuron.

Just as the brain updates its neural connections through experience, so too the weight matrix $W_1$ and the bias vector $b_1$ are learned during training.

The learned arrays don't have a neat visual interpretation like the embedding matrix, but for what it's worth, here they are printed out.

In [ ]:
p.W1  # weight matrix

In [ ]:
p.b1  # bias vector

# Activation function

Before passing on to the second dense layer, the output first goes through a so-called _activation function_.  This is, so far, the only source of nonlinearity in the model.  Again, it has its origin in the biological analogy of the brain, corresponding to a neuron only "firing" if its input signal is sufficiently high.  So originally, activation functions were designed to cutoff at zero for large negative inputs, gradually ramp up and then saturate at one for large positive inputs.  The prototypical example was the _sigmoid_ function
$$
\textrm{sigmoid}(t) = \frac{1}{1 + e^{-t}}
$$
shown here.

In [ ]:
fig, ax = lines(-5..5, sigmoid, label="sigmoid")
axislegend(ax, position=:lt)
fig

# Beyond sigmoid

Gradually it was realised that many other kinds of nonlinearities could work well in different circumstances, and the (already somewhat tenuous) link to actual brain function was de-emphasised in the community.  These days deep learning literature is full of ideas for various choices of activation functions. Here we've kept things simple, and reached for an old favourite in the form of tanh. Compared to sigmoid, tanh has the advantage of being zero-centred, and having a steeper slope for values near zero, both of which are observed to improve training efficiency for models such as ours.

In [ ]:
lines!(-5..5, tanh, label="tanh")
axislegend(ax, position=:lt)
fig

# Importance of nonlinearity

Note that while it may not matter so much which particular form of nonlinearity you use, _some_ form of nonlinearity between dense layers is essential.  If it was all just matrix-vector multiplies and vector additions, the entire neural net would be a linear operation, severely limiting its ability to model complex nonlinear relationships.

Anyway, here's the result of applying tanh element-wise to the output of the dense layer.  You can see many of the activations saturating near to $\pm 1$, corresponding to large magnitude values of $z_1$.  If we were taking this model more seriously, it might be of interest to analyse which neurons were "firing" for this particular input, which were "negatively firing" and which were inactive, as a way of trying to understand what exactly the first dense layer of the model actually learned.


In [ ]:
h1 = tanh.(z1)

# Second dense layer

On to the second, and final, dense layer. This functions exactly like the first dense layer, in that it has a weight matrix $W_2$ and bias vector $b_2$ that were learned during training.

$$
z_2 = W_2\,  h_1 + b_2\,.
$$


Since this is the final dense layer, and we know that the final output of our model needs to be a vector of $|\mathcal{V}| = 27$ probabilities, the size of these parameters is already fully determined: namely $W_2 \in \mathbb{R}^{27 \times 16}$ and $b_2 \in \mathbb{R}^{27}$.  Again, for what it's worth, here are the learned values of these two parameters.

In [ ]:
p.W2

In [ ]:
p.b2

## Output of layer

The output of this dense layer is then calculated the same way as for the first.

There is a piece of terminology associated with this vector $z_2$: it's the vector of **logits**.  "Logits" is just the name for the input to the `softmax` function (to be described next), and that's exactly what we plan to do with this $z_2$.

In [ ]:
z2 = p.W2 * h1 + p.b2

# Softmax output

The final line of the model code uses the so-called `softmax` function:

In [ ]:
ŷ = softmax(z2)

# Derivation of softmax

What is this "softmax" function and why do we need it?  Remember that the output of this model needs to be a vector of _probabilities_.  And probabilities obey two rules that we absolutely must enforce: they are non-negative and they sum to one.

If you had to invent a function that took in a vector $z$ (the logits) and returned a new vector $y$ that was like $z$ except that every entry was a probability, then softmax is very likely the function you would invent.  Simply, it starts by taking the exponential of all the values in $z$, thereby ensuring they are positive. Then it divides the resulting vector by the sum of all the entries, thereby ensuring the total sum is one.

Hence, if $y = \textrm{softmax}(z)$, then
$$
y_i = \frac{e^{z_i}}{\sum_j e^{z_j}}\,.
$$

Besides our activation function (tanh), this is the only other nonlinear operation in the model.  Notice that, unlike the activation, the softmax function is truly a function of the whole vector $z$, not merely an element-wise calculation ($y_i$ depends on all of $z$, not just on $z_i$).

A note on terminology: "softmax" is famously a misnomer.  It should have been called softargmax.  Because that's what it is, a "soft" or smoothed, version of argmax, in the form of a probability vector concentrated on the maximising index.  You could even imagine a generalised version that uses a "temperature" factor $T$ in the formula:
$$
y_i = \frac{e^{z_i/T}}{\sum_j e^{z_j/T}}\,.
$$
In the limit $T \to 0$ this approaches argmax; i.e. whichever entry of $z$ is largest gets assigned probability 1, and everything else gets 0 (try it!).



# Generating new text

To actually use our model to generate new text, we follow the process:

* Take a text sequence as input, like `"alice said"`, and convert to character indices.
* Run the model on this sequence to get the probabilities of the next character.
* Sample the next character according to these probabilities.
* Append the new character to the end of the text sequence.
* Run the model on this new sequence, omitting the first character but including the new character you just generated (remember the context size is fixed).
* And so on.

Here's a function that does exactly that.

In [ ]:
# Generate some text with the trained model
function generate_text(model, context, nchars)

    sequence_length = length(context)

    # The result begins with the context
    result = context

    # Convert the start text to character indices
    input = string_to_idxs(context)

    # Generate new characters one by one
    for i = 1:nchars

        # Get the model's probability predictions for the next character
        probs = model(input)

        # Sample the next character based on the probabilities
        next_char_idx = sample(1:vocab_size, Weights(probs))

        # Append the character to the result, which becomes part of the input for the next iteration
        result *= idx_to_char(next_char_idx)
        input = string_to_idxs(result[end-sequence_length+1:end])
    end

    return result

end

## A few example runs

Let's give the model a few tries and see what we get.  Because we are using random sampling to choose the next character, the results will be different on each run.

While the output is hardly comparable to a line of text you'd find in the original novel, it has at least a resemblance to real English text.

In [ ]:
for i = 1:5
  newtext = generate_text(model, "alice said", 50)
  println(newtext)
end

# Conclusion

And that's it!  A full description of our simple model for generating new English text by extending an input sequence of text one character at a time.

We've covered:
* representing text as a sequence of character indices from a finite vocabulary
* the idea of a model parameterised by various matrices and vectors
* the specific notion of a neural network as such a model
* mapping character indices to embedding vectors
* dense (fully connected) layers with nonlinear activation functions
* converting ouput scores (logits) into probabilities using softmax
* the notion of "temperature" in the sampling process
* generating new text by repeatedly sampling a character and appending it to the context

But we're not done with this story.  In fact, there remain many questions still to be answered. Most notably, how is this model actually trained?  Remember for this lesson we just loaded a set of pre-trained parameters $p$. But where did they come from?

Answering this question, and getting to the point where we are able to train the model ourselves from scratch, will occupy our attention for the next few lessons.